# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianLogisticRegression, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.34893695  0.19688423  0.91186718 -0.06005952  0.00401736]
 [ 0.99976898 -0.31903798 -0.53716279 -0.60025315  0.33776753]
 [-0.43632653  0.23500865 -0.33669263  0.40982046  0.08591975]
 [ 0.53461827 -0.47902015 -0.35288617 -0.0199405  -0.35625943]
 [ 0.44022825 -0.61584435  0.26902483 -0.72442012 -0.81711553]
 [ 0.3551328   0.54307667 -0.11291123  0.48187203  0.62740013]
 [-0.84701573  0.10031094  0.51946317 -0.53188947  0.65233626]
 [ 0.9534233  -0.90340091 -0.45135071  0.12761641  0.47940979]
 [-0.60732955 -0.95589278 -0.67009613 -0.49869096 -0.32123924]
 [-0.89528708 -0.2652834   0.63693658 -0.69452743 -0.7099346 ]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a1', 'a1', 'a1', 'a1', 'a2', 'a1', 'a2', 'a1', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 0, 0, 1, 1, 1, 0, 1, 1, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:33,  1.02s/it]

SVI:   3%|▎         | 1/34 [00:01<00:33,  1.02s/it, loss=1366.3889]

SVI:   6%|▌         | 2/34 [00:01<00:32,  1.02s/it, loss=1566.7321]

SVI:   9%|▉         | 3/34 [00:01<00:31,  1.02s/it, loss=1308.3618]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.19it/s, loss=1132.8319]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.19it/s, loss=941.0307] 

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.19it/s, loss=1009.5261]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.14it/s, loss=1009.5261]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.14it/s, loss=1069.3500]